##Small GPT

Блокнот, в котором будет реализован и обучен компактный `decoder-only` трансформер. Здесь будут приведены все необходимые для этого функции, в итоговой версии используется `Flash-Attention` механизм, его реализация здесь приведена не будет.

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from tqdm.notebook import tqdm
from torch.utils.data import Subset, DataLoader

In [2]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout_prob=0.0, causal=True):
        assert d_model % num_heads == 0
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.dim = d_model // num_heads
        self.causal_flag = causal
        self.dropout_prob = dropout_prob

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

        self.dropout_layer = nn.Dropout(dropout_prob)

    def forward(self, X, padding_mask=None):
        batch_size, t, _ = X.shape

        Q = self.Wq(X)
        K = self.Wk(X)
        V = self.Wv(X)

        Q = Q.view(batch_size, t, self.num_heads, self.dim).transpose(1, 2)
        K = K.view(batch_size, t, self.num_heads, self.dim).transpose(1, 2)
        V = V.view(batch_size, t, self.num_heads, self.dim).transpose(1, 2)

        #Паддинг
        attn_mask = None
        if padding_mask is not None:
            pad_mask = padding_mask.unsqueeze(1).unsqueeze(2)  #(B, 1, 1, T)
            attn_mask = pad_mask.float().masked_fill(pad_mask == 0, float('-inf'))

        #Flash
        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            dropout_p=self.dropout_prob if self.training else 0.0,
            is_causal=self.causal_flag
        )

        out = out.transpose(1, 2).contiguous().view(batch_size, t, self.d_model)
        out = self.Wo(out)
        out = self.dropout_layer(out)

        return out

Комментарий: если бы мы не использовали Flash Attention, то надо было бы также написать функцию для создания `causal_mask`. Она бы выглядела примерно так:


```
torch.triu(torch.ones(size, size) * -float('inf'))
```

Вычисление score:


```
scores = Q@K.transpose(-2, -1)/torch.sqrt(self.d_k_tensor)
O = F.softmax(scores + MASK, dim=-1)@V

out = self.Wo(O.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model))
```




In [3]:
class FeedForward(nn.Module):
    def __init__(self, d_model, expansion_factor=4, dropout=0.0, bias=True):
        super().__init__()

        self.fc1 = nn.Linear(d_model, expansion_factor * d_model, bias=bias)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(expansion_factor * d_model, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x

In [4]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads=4, expansion_factor=4, dropout=0.2, bias=True, causal=True):
        super().__init__()

        self.d_model = d_model

        self.layer_norm1 = nn.LayerNorm(d_model)
        self.att = MultiHeadSelfAttention(d_model, num_heads, dropout, causal)
        self.layer_norm2 = nn.LayerNorm(d_model)
        self.ff_layer = FeedForward(d_model, expansion_factor, dropout, bias)


    def forward(self, x, padding_mask=None):
        x = x + self.att(self.layer_norm1(x), padding_mask=padding_mask)
        x = x + self.ff_layer(self.layer_norm2(x))

        return x

Собираем модули воедино, добавляем позиционное кодирование, правильную инциализацию весов и полносвязный слой в конце.

In [5]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, n_layer, block_size, dropout=0.0, causal=True, expansion_factor=4):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, expansion_factor, dropout, True, causal)
            for _ in range(n_layer)
        ])

        self.layer_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init_weights)

    #from karpathy
    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, idx, padding_mask=None):
        batch_size, t = idx.shape

        assert t <= self.block_size

        token_embed = self.token_embedding(idx)
        pos = torch.arange(0, t, device=idx.device).unsqueeze(0)
        pos_embed = self.position_embedding(pos)

        x = self.dropout(token_embed + pos_embed)

        for block in self.blocks:
            x = block(x, padding_mask)

        x = self.layer_norm(x)

        return self.head(x)

    # авторегрессионная генерация.
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:] if idx.shape[1] > self.block_size else idx
            logits = self(idx_cond)[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


###Data Prep

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import GPT2Tokenizer

In [7]:
def tokenize_function(examples, tokenizer, block_size):
        return tokenizer(
            examples["lyrics"],
            truncation=True,
            max_length=block_size,
            padding=False,
            return_attention_mask=True,
            )


def collate_fn(batch):
        input_ids = [item["input_ids"] for item in batch]
        attention_mask = [item["attention_mask"] for item in batch]

        max_len = max(len(ids) for ids in input_ids)

        padded_input_ids = []
        padded_attention_mask = []
        for ids, mask in zip(input_ids, attention_mask):
            pad_len = max_len - len(ids)
            padded_input_ids.append(ids + [tokenizer.pad_token_id] * pad_len)
            padded_attention_mask.append(mask + [0] * pad_len)

        return {
            "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(padded_attention_mask, dtype=torch.float),
            }

In [9]:
from functools import partial

dataset = load_dataset('theelderemo/genius-lyrics-cleaned')

README.md:   0%|          | 0.00/26.1k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  326MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  279MB            

data/train-00001-of-00010.parquet: downloading bytes:           |  0.00B            

...

data/train-00008-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  241MB            

data/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  224MB            

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3179588 [00:00<?, ? examples/s]

In [10]:
new_dataset = dataset['train'].select(range(50000))

In [11]:
new_dataset = new_dataset.train_test_split(test_size=0.2)

In [12]:
new_dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'artist', 'tag', 'year', 'lyrics'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['title', 'artist', 'tag', 'year', 'lyrics'],
        num_rows: 10000
    })
})

In [13]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

batch_size = 64
train_split = 'train'
val_split = 'test'
num_proc = 1
block_size = 128

tokenize_function_part = partial(tokenize_function, tokenizer=tokenizer, block_size=block_size)

tokenized_dataset = new_dataset.map(
        tokenize_function_part,
        batched=True,
        num_proc=num_proc,
        remove_columns=dataset["train"].column_names,
        )

train_loader = DataLoader(
        tokenized_dataset[train_split],
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0
        )

valid_loader = DataLoader(
        tokenized_dataset[val_split],
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0
        )

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

###Train pipeline

In [14]:
def train(model, optimizer, train_loader, valid_loader, tokenizer, num_epochs=5, device='cuda'):
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        losses = []

        loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False)
        for batch in loop:
            optimizer.zero_grad()

            X = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)

            logits = model(X, padding_mask=mask)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)), X[:, 1:].reshape(-1))
            losses.append(loss.cpu().item())

            loss.backward()
            optimizer.step()
        print(f'Num epoch: {epoch}, Train Loss: {np.mean(losses)}')

        valid(model, valid_loader, tokenizer)

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
        }, f'checkpoint_epoch_{epoch+1}.pt')


def valid(model, valid_loader, tokenizer, device='cuda'):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in valid_loader:
            X = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)

            logits = model(X, padding_mask=mask)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)), X[:, 1:].reshape(-1))

            losses.append(loss.cpu().item())

    print(f'Valid loss: {np.mean(losses)}')
    print('Autoregressive generation example:')
    prompt = "Kind sir Kovka once said"
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=model.block_size)
    input_ids = encoded['input_ids'].to(device)

    with torch.no_grad():
        generated = model.generate(
            idx=input_ids,
            max_new_tokens=50,
            temperature=0.8,
            top_k=40
        )

    generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    print(generated_text)
    print('-' * 50)

###Train & Valid Process

In [15]:
model = GPT(tokenizer.vocab_size, d_model=256, num_heads=4, n_layer=2, block_size=128, dropout=0.2, causal=True, expansion_factor=4)

In [17]:
optimizer = optim.AdamW(params=model.parameters(), lr=1e-4, weight_decay=5e-2)

In [18]:
train(model, optimizer, train_loader, valid_loader, tokenizer, num_epochs=5, device='cuda')

Epoch 1/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 0, Train Loss: 3.5700692207336426
Valid loss: 3.6188873940972006
Autoregressive generation example:
Kind sir Kovka once said

I said, "Mama said "It's been a couple"
I wouldn't be a dog?"
I got a bitch

So I got a little bitch
I heard that
I got a bad bitch
I got
--------------------------------------------------


Epoch 2/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 1, Train Loss: 3.5261022300720213
Valid loss: 3.591339594239642
Autoregressive generation example:
Kind sir Kovka once said: "I know"

I told you "I heard you're?"
I said "What are you?"
And I know
So I'm still doing you?"
I'm not sorry
You're just a fool

[
--------------------------------------------------


Epoch 3/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 2, Train Loss: 3.4913092414855957
Valid loss: 3.5685957996708573
Autoregressive generation example:
Kind sir Kovka once said

The truth was a tale
And the first man was raised
He was a tale of children
And he was a slave
Then he saw him he was a father
He were a man in his back man
He was a dog
--------------------------------------------------


Epoch 4/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 3, Train Loss: 3.4601842948913575
Valid loss: 3.5502937903070144
Autoregressive generation example:
Kind sir Kovka once said

Oh dear
I knew it was just something, but I was still here to cry
I wish it was right here
The one that was wrong

When I knew it was
But I knew it was wrong
And I had
--------------------------------------------------


Epoch 5/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 4, Train Loss: 3.4303403884887693
Valid loss: 3.533677236289735
Autoregressive generation example:
Kind sir Kovka once said

There’s no one's a way

I know how it works
But the way it feels

I know what it means
But I don’t seem to
What it seems?

I know what
--------------------------------------------------


In [20]:
train(model, optimizer, train_loader, valid_loader, tokenizer, num_epochs=5, device='cuda')

Epoch 1/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 0, Train Loss: 3.2853984134674072
Valid loss: 3.47172151705262
Autoregressive generation example:
Kind sir Kovka once said
"My father said
"Dear God."
I'm not scared to have a little sister"
And the world seemed to be alive
And the world was alive
My dear friend said goodbye

Oh, I'm not afraid to
--------------------------------------------------


Epoch 2/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 1, Train Loss: 3.265338582611084
Valid loss: 3.469017744064331
Autoregressive generation example:
Kind sir Kovka once said "Ah oh"

Well, he said "Bobby"
Well, it's time to begin to begin"
And he said, "Oh, who, who, who, who, who, who, who, who, who
--------------------------------------------------


Epoch 3/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 2, Train Loss: 3.2462917461395264
Valid loss: 3.466328675579873
Autoregressive generation example:
Kind sir Kovka once said
So the voice that did
And I guess I can't believe
And I guess I wouldn't believe
I don't believe it
I know you'll never believe

I still believe it

And I know you're not the
--------------------------------------------------


Epoch 4/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 3, Train Loss: 3.227680479812622
Valid loss: 3.4620192233164597
Autoregressive generation example:
Kind sir Kovka once said, "I can't be sorry."

No matter what it is
But no matter what time is
Because I can't be okay and no matter what I wanna say
It's a matter what I want
That's a matter what
--------------------------------------------------


Epoch 5/5:   0%|          | 0/625 [00:00<?, ?it/s]

Num epoch: 4, Train Loss: 3.210489111328125
Valid loss: 3.4587345730726886
Autoregressive generation example:
Kind sir Kovka once said
Cuz he doesn’t mean
I’d like to have a choice
And there’s a part of the past
There’s a meaning of what
There’s a way to end
I
--------------------------------------------------


Проблемы:
1. В настоящее время кол-во уникальных токенов\кол-во параметров $\approx 10$,модель можно попробовать увеличить и действовать по стратегиям, на которые опирались при обучении ранних GPT 2020 года.

2. Проблема с малым `block_size = 128`. От текста остаются лишь первые `block_size` токенов, поэтому никак не улавливается структура. Надо стремиться увеличить этот параметр.

3. Пока используется лишь малая часть датасета.